# RC-DCLP: Final accepted model (E4-on-E3)

**What is included?** Five original trained structural-subgraph GNN checkpoints, five Double-DQN controller checkpoints, the final retention-aware trainer, representative replay + historical distillation, the actual DBLP logs, and matched-control analysis.

**Not included in the final method:** multi-trajectory DQN (later E5) or encoder-LR intervention (E6b).

Run the cells in order. The archived weights can be verified **without any dataset**. To reproduce frozen 2015–2016 predictions or retrain, supply your original DBLP CSV with SHA256 `3cade711abbe2452f1bf22b1ed938ed46640d58651114676e85aea4198578744`. The source ZIP does **not** contain that dataset.

## 1 — Upload and extract the final project ZIP

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import os, shutil, subprocess, sys
from google.colab import files, drive

drive.mount('/content/drive')
archive = Path('/content/RC_DCLP_FINAL_Complete.zip')
if not archive.is_file():
    print('Upload RC_DCLP_FINAL_Complete.zip')
    uploaded = files.upload()
    assert archive.name in uploaded, 'Upload the exact complete ZIP'
    archive.write_bytes(uploaded[archive.name])
with ZipFile(archive) as z:
    assert z.testzip() is None, 'Archive integrity failed'
    z.extractall('/content')
project = Path('/content/RC_DCLP_FINAL')
assert (project/'scripts/rc_dclp.py').is_file(), 'Unexpected ZIP folder layout'
os.chdir(project)
print('Project ready:', project)

## 2 — Install dependencies and verify actual trained weights

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[test]'],check=True)
subprocess.run([sys.executable,'scripts/rc_dclp.py','verify'],check=True)
subprocess.run([sys.executable,'-m','pytest','-q',
                'tests/test_final_release.py',
                'tests/test_e6g_checkpoint.py::test_e4_retention_constraint_and_frozen_final_years',
                'tests/test_e6g_checkpoint.py::test_rejected_update_restores_model_and_optimizer'],check=True)

## 3 — Compare final model with the exact matched earlier RRD control

In [ ]:
subprocess.run([sys.executable,'scripts/rc_dclp.py','compare',
                '--output','outputs/paired_comparison'],check=True)

## 4 — Find the original DBLP CSV (needed for prediction/retraining)

In [ ]:
import hashlib
expected_sha='3cade711abbe2452f1bf22b1ed938ed46640d58651114676e85aea4198578744'
candidates = [
 Path('/content/dblp_tgat_5000.csv'),
 Path('/content/drive/MyDrive/Graph_RL_Project/datasets/dblp_tgat_5000.csv'),
 Path('/content/drive/MyDrive/Graph_RL_Project/dblp_tgat_5000.csv'),
]
csv = next((p for p in candidates if p.is_file()),None)
if csv is None:
    print('Original CSV not found in expected paths. Upload dblp_tgat_5000.csv to /content if you want prediction/retraining.')
else:
    observed=hashlib.sha256(csv.read_bytes()).hexdigest()
    assert observed==expected_sha, f'DBLP fingerprint mismatch: {observed}'
    print('Verified exact dataset:',csv)

## 5 — Recompute frozen test predictions from the original trained seed-7 model

In [ ]:
if csv is None:
    print('Skipped: supply the exact original DBLP CSV first.')
else:
    subprocess.run([sys.executable,'scripts/rc_dclp.py','predict',
                    '--csv',str(csv),'--seed','7','--years','2015,2016',
                    '--evaluate','--output','outputs/seed7_frozen_scores.csv'],check=True)

## 6 — Optional: retrain all five seeds using the **exact final E4-on-E3 method**

Disabled by default. If enabled, the script runs the final model only, with no E5 or E6b modifications. It can be computationally substantial; reuse the **same protocol** for the seed-7 pilot and five-seed continuation. Experiment locks prevent mismatched resumed runs.

In [ ]:
RUN_FULL_RETRAIN = False
if RUN_FULL_RETRAIN:
    assert csv is not None, 'Original DBLP CSV is required'
    import torch
    device='cuda' if torch.cuda.is_available() else 'cpu'
    for seed_group in ('7','7,13,42,101,202'):
        subprocess.run([sys.executable,'scripts/rc_dclp.py','train',
                        '--csv',str(csv),'--seeds',seed_group,
                        '--device',device,'--results-dir','runs/final'],check=True)
    subprocess.run([sys.executable,'scripts/rc_dclp.py','compare',
                    '--results-dir','runs/final','--output','outputs/new_five_seed_analysis'],check=True)
else:
    print('Original trained weights and recorded results preserved. Full retraining skipped.')

## 7 — Save outputs to Drive and download a result ZIP

In [ ]:
from zipfile import ZIP_DEFLATED
save_root=Path('/content/drive/MyDrive/Graph_RL_Project/FINAL_RC_DCLP')
save_root.mkdir(parents=True,exist_ok=True)
result_zip=save_root/'RC_DCLP_Results_And_Verification.zip'
with ZipFile(result_zip,'w',compression=ZIP_DEFLATED,compresslevel=3,allowZip64=True) as z:
    for folder in (project/'outputs',project/'runs'):
        if folder.exists():
            for path in folder.rglob('*'):
                if path.is_file():z.write(path,arcname=str(path.relative_to(project)))
    for path in (project/'recorded'/'e6g_checkpoint_analysis').glob('*.csv'):
        z.write(path,arcname=str(path.relative_to(project)))
print('Saved result ZIP:',result_zip,'size MB:',round(result_zip.stat().st_size/2**20,2))
files.download(str(result_zip))